# Train CatBoost Model Using SPARCS 2024

This notebook trains a CatBoost model to predict prolonged length of stay using the 2024 SPARCS inpatient discharge dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# catboost - for train CatBOOST model
# shap - for XAI use
# joblib - for save pipeline

!pip install -q catboost shap joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from catboost import CatBoostClassifier

In [ ]:
BASE_DIR = "/content/drive/MyDrive/FYP/SPARCS"

PROCESSED_PATH = f"{BASE_DIR}/processed/sparcs_2024_processed.csv"
SPLIT_DIR = f"{BASE_DIR}/splits"
MODEL_DIR = f"{BASE_DIR}/models"
RESULT_DIR = f"{BASE_DIR}/results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
# Load Processed Dataset and Split Files

df = pd.read_csv(PROCESSED_PATH, low_memory=False)

train_idx = np.load(f"{SPLIT_DIR}/train_idx.npy")
val_idx = np.load(f"{SPLIT_DIR}/val_idx.npy")
test_idx = np.load(f"{SPLIT_DIR}/test_idx.npy")

feature_columns = joblib.load(f"{SPLIT_DIR}/feature_columns.joblib")
feature_schema = joblib.load(f"{SPLIT_DIR}/feature_schema.joblib")

print("Dataset shape:", df.shape)
print("Number of selected features:", len(feature_columns))
print(feature_columns)

Dataset shape: (2196737, 35)
Number of selected features: 13
['Age Group', 'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'CCSR Diagnosis Description', 'CCSR Procedure Description', 'APR DRG Description', 'APR MDC Description', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Emergency Department Indicator']


In [ ]:
# Prepare X and y

target_col = "prolonged_los"

X = df[feature_columns].copy()
y = df[target_col].copy()

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

X_val = X.iloc[val_idx]
y_val = y.iloc[val_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (1405911, 13) (1405911,)
Validation: (351478, 13) (351478,)
Test: (439348, 13) (439348,)


In [ ]:
# Check Class Distribution

class_distribution = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    "positive_count": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "total_count": [len(y_train), len(y_val), len(y_test)]
})

class_distribution

,split,positive_rate,positive_count,total_count
0,train,0.241135,339015,1405911
1,validation,0.241136,84754,351478
2,test,0.241137,105943,439348


In [ ]:
# Identify Categorical and Numeric Columns
# Most of the features selected by SPARCS are categorical
# Will continue to use the One-Hot Encoder, maintaining consistency with XGBoost/LightGBM


cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

Categorical columns: ['Age Group', 'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'CCSR Diagnosis Description', 'CCSR Procedure Description', 'APR DRG Description', 'APR MDC Description', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Emergency Department Indicator']
Numeric columns: []


In [ ]:
# Build Preprocessing Pipeline

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [ ]:
# Handle Class Imbalance
# CatBoost can use class weights. It can calculate class weights manually

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count

negative_weight = 1.0
positive_weight = negative_count / positive_count

class_weights = [negative_weight, positive_weight]

print("Positive count:", positive_count)
print("Negative count:", negative_count)
print("Class weights:", class_weights)

Positive count: 339015
Negative count: 1066896
Class weights: [1.0, np.float64(3.1470465908588117)]


In [ ]:
# Train CatBoost Model

cat_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    class_weights=class_weights,
    random_seed=42,
    verbose=False
)

cat_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", cat_model)
])

cat_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  []),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Age Group', 'Gender',
                                                   'Race', 'Ethnicity',
                                                   'Type of Admission',
                                                   'CCSR Diagnosis Description',...
                                                   'APR DRG Description',
                                                   'APR MDC Description',
                                                   'APR Severity of Illness '
                                                   'Description',
                                                   'APR Risk of Mortality',
                                                   'APR Medical Surgical '
                                                   'Description',
                                                   'Emergency Department '
                                                   'Indicator'])])),
                ('model',
                 CatBoostClassifier(class_weights=[1.0, np.float64(3.1470465908588117)], depth=6, eval_metric='AUC', iterations=500, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=False))])

In [ ]:
# Evalution Function

def evaluate_binary_classifier(model, X_data, y_true, split_name, threshold=0.5):
    y_prob = model.predict_proba(X_data)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "model": "CatBoost",
        "dataset": "SPARCS 2024",
        "split": split_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

    return metrics, y_prob, y_pred

In [ ]:
# Validate Model

val_metrics, y_prob_val, y_pred_val = evaluate_binary_classifier(
    cat_pipeline,
    X_val,
    y_val,
    "validation",
    threshold=0.5
)

pd.DataFrame([val_metrics])

,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,CatBoost,SPARCS 2024,validation,0.5,0.869825,0.687936,0.77542,0.522228,0.806511,0.633959


In [ ]:
print(confusion_matrix(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

[[204188  62536]
 [ 16399  68355]]
              precision    recall  f1-score   support

           0       0.93      0.77      0.84    266724
           1       0.52      0.81      0.63     84754

    accuracy                           0.78    351478
   macro avg       0.72      0.79      0.74    351478
weighted avg       0.83      0.78      0.79    351478



In [ ]:
# Threshold Tuning

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.05):
    metrics, _, _ = evaluate_binary_classifier(
        cat_pipeline,
        X_val,
        y_val,
        "validation",
        threshold=threshold
    )
    threshold_results.append(metrics)

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(by="f1", ascending=False).head(10)

,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
8,CatBoost,SPARCS 2024,validation,0.60,0.869825,0.687936,0.813203,0.595169,0.704639,0.645294
7,CatBoost,SPARCS 2024,validation,0.55,0.869825,0.687936,0.799458,0.563008,0.752130,0.643971
9,CatBoost,SPARCS 2024,validation,0.65,0.869825,0.687936,0.822527,0.626802,0.652524,0.639404
6,CatBoost,SPARCS 2024,validation,0.50,0.869825,0.687936,0.775420,0.522228,0.806511,0.633959
5,CatBoost,SPARCS 2024,validation,0.45,0.869825,0.687936,0.754926,0.495181,0.838934,0.622771
10,CatBoost,SPARCS 2024,validation,0.70,0.869825,0.687936,0.830146,0.673135,0.574651,0.620007
4,CatBoost,SPARCS 2024,validation,0.40,0.869825,0.687936,0.729724,0.467567,0.871074,0.608506
3,CatBoost,SPARCS 2024,validation,0.35,0.869825,0.687936,0.697537,0.438178,0.901291,0.589675
11,CatBoost,SPARCS 2024,validation,0.75,0.869825,0.687936,0.831250,0.719018,0.492744,0.584755
2,CatBoost,SPARCS 2024,validation,0.30,0.869825,0.687936,0.658690,0.408605,0.928629,0.567503


# Select Deployment Threshold

The threshold with the best validation F1-score is identified for reference.  
The final deployment threshold is selected using the same rule applied to XGBoost and LightGBM: prioritize recall for prolonged LOS screening, then choose the best F1-score among suitable thresholds.

In [ ]:
best_f1_threshold_row = threshold_df.sort_values(by="f1", ascending=False).iloc[0]
best_f1_threshold = float(best_f1_threshold_row["threshold"])

best_f1_threshold_row

,8
model,CatBoost
dataset,SPARCS 2024
split,validation
threshold,0.6
roc_auc,0.869825
pr_auc,0.687936
accuracy,0.813203
precision,0.595169
recall,0.704639
f1,0.645294


In [ ]:
candidate_thresholds = threshold_df[threshold_df["recall"] >= 0.70]

if len(candidate_thresholds) > 0:
    deployment_threshold_row = candidate_thresholds.sort_values(
        by=["f1", "precision"],
        ascending=False
    ).iloc[0]
else:
    deployment_threshold_row = best_f1_threshold_row

deployment_threshold = float(deployment_threshold_row["threshold"])

deployment_threshold_row

,8
model,CatBoost
dataset,SPARCS 2024
split,validation
threshold,0.6
roc_auc,0.869825
pr_auc,0.687936
accuracy,0.813203
precision,0.595169
recall,0.704639
f1,0.645294


In [ ]:
# Final Validation and Test Evaluation

val_metrics_deploy, y_prob_val, y_pred_val = evaluate_binary_classifier(
    cat_pipeline,
    X_val,
    y_val,
    "validation",
    threshold=deployment_threshold
)

test_metrics_deploy, y_prob_test, y_pred_test = evaluate_binary_classifier(
    cat_pipeline,
    X_test,
    y_test,
    "test",
    threshold=deployment_threshold
)

metrics_df = pd.DataFrame([val_metrics_deploy, test_metrics_deploy])
metrics_df

,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,CatBoost,SPARCS 2024,validation,0.6,0.869825,0.687936,0.813203,0.595169,0.704639,0.645294
1,CatBoost,SPARCS 2024,test,0.6,0.869195,0.686553,0.813027,0.595167,0.702378,0.644343


In [ ]:
print("Validation confusion matrix")
print(confusion_matrix(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

print("Test confusion matrix")
print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

Validation confusion matrix
[[226102  40622]
 [ 25033  59721]]
              precision    recall  f1-score   support

           0       0.90      0.85      0.87    266724
           1       0.60      0.70      0.65     84754

    accuracy                           0.81    351478
   macro avg       0.75      0.78      0.76    351478
weighted avg       0.83      0.81      0.82    351478

Test confusion matrix
[[282790  50615]
 [ 31531  74412]]
              precision    recall  f1-score   support

           0       0.90      0.85      0.87    333405
           1       0.60      0.70      0.64    105943

    accuracy                           0.81    439348
   macro avg       0.75      0.78      0.76    439348
weighted avg       0.83      0.81      0.82    439348



In [ ]:
# Summary Check

positive_baseline = y_train.mean()

summary_check = pd.DataFrame([
    {
        "metric": "Positive baseline",
        "value": positive_baseline,
        "requirement": "PR-AUC should be higher than this"
    },
    {
        "metric": "Validation ROC-AUC",
        "value": val_metrics_deploy["roc_auc"],
        "requirement": ">= 0.70"
    },
    {
        "metric": "Validation PR-AUC",
        "value": val_metrics_deploy["pr_auc"],
        "requirement": "> positive baseline"
    },
    {
        "metric": "Validation Recall",
        "value": val_metrics_deploy["recall"],
        "requirement": ">= 0.65 preferred"
    },
    {
        "metric": "Test ROC-AUC",
        "value": test_metrics_deploy["roc_auc"],
        "requirement": ">= 0.70"
    },
    {
        "metric": "Test PR-AUC",
        "value": test_metrics_deploy["pr_auc"],
        "requirement": "> positive baseline"
    },
    {
        "metric": "Test Recall",
        "value": test_metrics_deploy["recall"],
        "requirement": ">= 0.65 preferred"
    }
])

summary_check

,metric,value,requirement
0,Positive baseline,0.241135,PR-AUC should be higher than this
1,Validation ROC-AUC,0.869825,>= 0.70
2,Validation PR-AUC,0.687936,> positive baseline
3,Validation Recall,0.704639,>= 0.65 preferred
4,Test ROC-AUC,0.869195,>= 0.70
5,Test PR-AUC,0.686553,> positive baseline
6,Test Recall,0.702378,>= 0.65 preferred


In [ ]:
# Risk Level Mapping

def risk_level(probability):
    if probability < 0.33:
        return "Low"
    elif probability < 0.66:
        return "Medium"
    else:
        return "High"

In [ ]:
sample_output = pd.DataFrame({
    "predicted_probability": y_prob_test[:10],
    "prolonged_los_prediction": (y_prob_test[:10] >= deployment_threshold).astype(int),
    "risk_level": [risk_level(p) for p in y_prob_test[:10]]
})

sample_output

,predicted_probability,prolonged_los_prediction,risk_level
0,0.484833,0,Medium
1,0.166947,0,Low
2,0.292094,0,Low
3,0.314606,0,Low
4,0.871590,1,High
5,0.021439,0,Low
6,0.866845,1,High
7,0.830860,1,High
8,0.148245,0,Low
9,0.136772,0,Low


In [ ]:
# Save Model, Metrics, Threshold Result, Metadata

cat_model_path = f"{MODEL_DIR}/catboost_sparcs_los_pipeline.joblib"
joblib.dump(cat_pipeline, cat_model_path)

metrics_df.to_csv(f"{RESULT_DIR}/catboost_sparcs_metrics.csv", index=False)
threshold_df.to_csv(f"{RESULT_DIR}/catboost_sparcs_threshold_tuning.csv", index=False)

metadata = {
    "model_name": "CatBoost",
    "dataset": "SPARCS 2024",
    "target": "prolonged_los",
    "target_definition": "Length of Stay >= 7 days",
    "selected_threshold": deployment_threshold,
    "threshold_selection_reason": "Selected on validation set by prioritizing recall >= 0.70 and then F1-score.",
    "best_f1_threshold": best_f1_threshold,
    "risk_level_thresholds": {
        "low": "<0.33",
        "medium": "0.33-0.66",
        "high": ">=0.66"
    },
    "feature_columns": feature_columns,
    "feature_schema": feature_schema
}

joblib.dump(metadata, f"{MODEL_DIR}/catboost_sparcs_metadata.joblib")

print("Saved CatBoost pipeline:", cat_model_path)
print("Saved metrics and metadata.")

Saved CatBoost pipeline: /content/drive/MyDrive/FYP/SPARCS/models/catboost_sparcs_los_pipeline.joblib
Saved metrics and metadata.


## Summary

A CatBoost model was trained using the selected SPARCS 2024 deployment-friendly features.  
The same train, validation, and test split used in the XGBoost and LightGBM experiments was reused to support fair model comparison.  
The model was trained with class weights to address prolonged LOS class imbalance.  
Threshold tuning was performed on the validation set, and the final threshold was selected by prioritizing recall for prolonged LOS screening.  
The trained pipeline, metrics, threshold tuning results, and metadata were saved for later model comparison and deployment.